<a href="https://colab.research.google.com/github/debayanpanda3-cloud/industrial-equipment-failure-forecasting/blob/main/Industrial_Equipment_Failure_Forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> ##  **Project Title:** Industrial Equipment Failure Forecasting
> ###  **Author:** Debayan Panda
---

##  1. Project at a Glance

| Item | Project Details |
| :--- | :--- |
| **Problem** | Forecasting machine failure from sensor and operating telemetry data |
| **Dataset** | AI4I 2020 Predictive Maintenance Dataset (10,000 records) |
| **Tools & Libraries** | Python, Google Colab, pandas, NumPy, Matplotlib, scikit-learn |
| **Machine Learning Models** | Logistic Regression, Multi-Layer Perceptron (MLP Classifier) |
| **Key Evaluation Metrics** | ROC-AUC, Class-wise Precision / Recall / F1-Score, Custom Financial Penalty |
| **Main Reported Result** | **MLP Test ROC-AUC: 0.971** (Best Tested Decision Threshold: 0.1) |

---

##  2. Introduction & Motivation

Predictive maintenance leverages historical and real-time equipment telemetry to identify complex operational patterns that precede machinery breakdowns. Traditional strategies rely either on rigid, scheduled inspections (which can cause unnecessary downtime) or reactive repairs (which risk severe operational disruption and financial loss).

By implementing predictive maintenance, inspection teams can prioritize high-risk equipment proactively. Machine learning models excel in this context because failure modes typically manifest through multi-variable interactions—such as subtle shifts across speed, torque, and thermal dynamics—rather than a single isolated alarm trigger.

---

##  3. Project Objectives

1. **Data Inspection & Cleaning:** Load and verify the dataset, complete missing-value checks, and perform one-hot encoding for categorical variables.
2. **Class Imbalance Analysis:** Analyze the ratio between normal operating conditions and rare failure events.
3. **Feature Engineering:** Construct domain-specific indicator variables, including operational temperature differentials and speed–torque interaction terms.
4. **Model Training & Validation:** Train and evaluate Logistic Regression and Multi-Layer Perceptron (MLP) architectures using 80/20 train–test splits and 5-fold cross-validation.
5. **Model Interpretation & Financial Optimization:** Apply permutation feature importance and test cost-sensitive decision thresholds to minimize custom financial misclassification penalties.

---

##  4. Workflow & Methodology

* **Data Ingestion & Cleaning:** Tabular inspection and validation using `pandas`.
* **Preprocessing & Scaling:** Feature scaling, missing-value verifications, and one-hot encoding.
* **Exploratory Data Analysis (EDA):** Quantile/IQR analysis, Pearson correlation matrices, and class distribution profiling.
* **Feature Engineering:** Creating domain-derived features (e.g., Temperature Difference, Speed–Torque Product).
* **Supervised Learning:** Training Baseline Logistic Regression and Multi-Layer Perceptron (MLP) Classifiers.
* **Validation & Interpretation:** Model assessment via ROC-AUC, permutation importance, and custom threshold optimization.

Question 1: Data Loading & Initial Inspection
Load the predictive maintenance dataset directly from the UCI Machine Learning Repository using the URL: https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv. Drop the 'UDI' and 'Product ID' columns, as they do not provide predictive value. Display the first five rows of the dataset.

Hint: Utilize standard tabular data loading functions. You may use the provided static URL or explore the modern ucimlrepo python package. Drop the specified string columns along the appropriate axis.

Expected Output: A tabular representation of the first five rows of the dataset excluding the identifier columns, displaying features related to air temperature, process temperature, and rotational speed.

In [ ]:
# Write your answer here

import pandas as pd

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv"

df = pd.read_csv(url)

df = df.drop(columns=["UDI", "Product ID"])

print(df.head())


  Type  Air temperature [K]  Process temperature [K]  Rotational speed [rpm]  \
0    M                298.1                    308.6                    1551   
1    L                298.2                    308.7                    1408   
2    L                298.1                    308.5                    1498   
3    L                298.2                    308.6                    1433   
4    L                298.2                    308.7                    1408   

   Torque [Nm]  Tool wear [min]  Machine failure  TWF  HDF  PWF  OSF  RNF  
0         42.8                0                0    0    0    0    0    0  
1         46.3                3                0    0    0    0    0    0  
2         49.4                5                0    0    0    0    0    0  
3         39.5                7                0    0    0    0    0    0  
4         40.0                9                0    0    0    0    0    0  


### Question 2: Missing Values & Data Type Conversion

Inspect the dataset for missing values across all columns. Subsequently, convert the 'Type' column (which contains categorical values: L, M, H) into numeric indicator features utilizing one-hot encoding. Integrate these new features into the dataset and remove the original 'Type' column.

**Hint:** Apply missing value aggregation methods to check for nulls. Use a categorical encoding technique to expand the 'Type' column into separate binary variables, ensuring the resulting columns are merged correctly with the primary data.

**Expected Output:** A printed series showing the aggregate count of missing values per column, followed by a preview of the modified dataset where the categorical 'Type' column has been replaced by binary columns.

In [ ]:
# Write your answer here

# Check missing values
print("Missing values before encoding:")
print(df.isnull().sum())

# Convert Type into separate numerical columns
df = pd.get_dummies(df, columns=["Type"], prefix="Type", dtype=int)

# Check missing values again
print("\nMissing values after encoding:")
print(df.isnull().sum())

# Display the first five rows
print("\nFirst five rows:")
print(df.head())


Missing values before encoding:
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64

Missing values after encoding:
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
Type_H                     0
Type_L                     0
Type_M                     0
dtype: int64

First five rows:
   Air temperature [K]  Process temperature [K]  Rotational speed [rpm]  \
0                298.1                    308.6                

### Question 3: Target Variable Distribution & Analytical Reasoning

Analyze the distribution of the target variable, 'Machine failure', by calculating the percentage of normal operations versus equipment failures. Following this calculation, explain why standard predictive accuracy is a mathematically flawed evaluation metric for this specific class distribution.

**Hint:** Compute the normalized frequency of each class in the target column and scale it to a percentage. Print your written answer using Python `print()` statements.

**Expected Output:** A printed numeric percentage breakdown of the two classes, followed by a printed paragraph detailing the analytical limitations of the accuracy metric in the presence of severe class imbalance.

In [ ]:
# Write your answer here

# Count normal operations and failures
counts = df["Machine failure"].value_counts()

# Calculate percentages
percentages = df["Machine failure"].value_counts(normalize=True) * 100

# Display the results
print("Number of machines in each class:")
print(counts)

print("\nPercentage of machines in each class:")
print(percentages)

# Explain why accuracy can be misleading
print("\nWhy accuracy can be misleading:")
print("The dataset is highly imbalanced, with 96.61% normal operations and only 3.39% failures. A model predicting all machines as normal would achieve 96.61% accuracy while missing every actual failure. Therefore, accuracy alone is misleading for evaluating failure detection.")


Number of machines in each class:
Machine failure
0    9661
1     339
Name: count, dtype: int64

Percentage of machines in each class:
Machine failure
0    96.61
1     3.39
Name: proportion, dtype: float64

Why accuracy can be misleading:
The dataset is highly imbalanced, with 96.61% normal operations and only 3.39% failures. A model predicting all machines as normal would achieve 96.61% accuracy while missing every actual failure. Therefore, accuracy alone is misleading for evaluating failure detection.


### Question 4: Feature Engineering - Thermodynamic and Mechanical Properties

Derive two new features to capture potential interactions. First, construct a 'Temperature_Difference' feature by subtracting 'Air temperature [K]' from 'Process temperature [K]'. Second, construct a 'Power_Estimate' feature by multiplying 'Rotational speed [rpm]' by 'Torque [Nm]'. Append both to your dataset.

**Hint:** Apply vectorized arithmetic operations between the existing numeric columns and assign the resulting vectors to new column identifiers.

**Expected Output:** A dataset preview demonstrating the inclusion of the two newly formulated numeric columns at the end of the variable list.

In [ ]:
# Write your answer here


# Calculate the temperature difference
df["Temperature_Difference"] = (
    df["Process temperature [K]"] - df["Air temperature [K]"]
)

# Estimate power using rotational speed and torque
df["Power_Estimate"] = (
    df["Rotational speed [rpm]"] * df["Torque [Nm]"]
)

# Display the complete dataset preview
print(df.head())



   Air temperature [K]  Process temperature [K]  Rotational speed [rpm]  \
0                298.1                    308.6                    1551   
1                298.2                    308.7                    1408   
2                298.1                    308.5                    1498   
3                298.2                    308.6                    1433   
4                298.2                    308.7                    1408   

   Torque [Nm]  Tool wear [min]  Machine failure  TWF  HDF  PWF  OSF  RNF  \
0         42.8                0                0    0    0    0    0    0   
1         46.3                3                0    0    0    0    0    0   
2         49.4                5                0    0    0    0    0    0   
3         39.5                7                0    0    0    0    0    0   
4         40.0                9                0    0    0    0    0    0   

   Type_H  Type_L  Type_M  Temperature_Difference  Power_Estimate  
0       0       0 

### Question 5: Outlier Detection using Interquartile Range

Implement an algorithm to identify outliers within the 'Rotational speed [rpm]' variable using the Interquartile Range (IQR) methodology. Calculate the precise upper and lower boundary thresholds, and return the absolute count of observations falling outside these statistical bounds. Do not delete these observations.

**Hint:** Programmatically compute the 25th percentile (Q1) and 75th percentile (Q3). Derive the IQR scalar. Formulate the lower bound as Q1 minus 1.5 times the IQR, and the upper bound as Q3 plus 1.5 times the IQR. Sum the instances matching this exclusion criteria.

**Expected Output:** A printed integer representing the total frequency of anomalous rotational speed observations exceeding the standard IQR boundaries.

In [ ]:
# Write your answer here

# Select the rotational speed column
speed = df["Rotational speed [rpm]"]

# Calculate the first and third quartiles
Q1 = speed.quantile(0.25)
Q3 = speed.quantile(0.75)

# Calculate the Interquartile Range
IQR = Q3 - Q1

# Calculate the lower and upper bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Count the outliers
outliers = speed[(speed < lower_bound) | (speed > upper_bound)]

# Display the results
print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Number of outliers:", len(outliers))

Q1: 1423.0
Q3: 1612.0
IQR: 189.0
Lower bound: 1139.5
Upper bound: 1895.5
Number of outliers: 418


### Question 6: Correlation Analysis & Feature Selection

Generate a Pearson correlation matrix encompassing all numeric variables. Extract and print the three variables demonstrating the strongest absolute linear correlation with the 'Machine failure' target. Exclude the distinct failure mode identifiers (TWF, HDF, PWF, OSF, RNF) and the target variable itself from this evaluation.

**Hint:** Compute the standard correlation matrix, apply a transformation to derive absolute magnitudes, isolate the column corresponding to the target, filter out the banned variables, sequence the results in descending order, and extract the top three.

**Expected Output:** A collection or series indicating the names of three continuous features paired with their absolute correlation coefficients relative to the target.

In [ ]:
# Write your answer here


# Calculate the Pearson correlation matrix
correlation_matrix = df.corr(numeric_only=True, method="pearson")

print("Pearson Correlation Matrix:")
print(correlation_matrix)

# Exclude the target variable and failure-type columns
excluded_columns = [
    "Machine failure", "TWF", "HDF", "PWF", "OSF", "RNF"
]

correlations = correlation_matrix["Machine failure"].drop(
    labels=excluded_columns,
    errors="ignore"
)

# Find the top 3 features using absolute correlation values
top_3 = correlations.loc[
    correlations.abs().sort_values(ascending=False).head(3).index
]

print("\nTop 3 features most correlated with Machine failure:")
print(top_3.abs())

Pearson Correlation Matrix:
                         Air temperature [K]  Process temperature [K]  \
Air temperature [K]                 1.000000                 0.876107   
Process temperature [K]             0.876107                 1.000000   
Rotational speed [rpm]              0.022670                 0.019277   
Torque [Nm]                        -0.013778                -0.014061   
Tool wear [min]                     0.013853                 0.013488   
Machine failure                     0.082556                 0.035946   
TWF                                 0.009955                 0.007315   
HDF                                 0.137831                 0.056933   
PWF                                 0.003470                -0.003355   
OSF                                 0.001988                 0.004554   
RNF                                 0.017688                 0.022279   
Type_H                             -0.023025                -0.017967   
Type_L                 

### Question 7: Data Splitting and Standardization (6 Marks)

Formulate the predictor matrix `X` by excluding 'Machine failure' and the specialized failure mode variables (TWF, HDF, PWF, OSF, RNF). Designate `y` as the 'Machine failure' target vector. Partition the dataset into 80% training and 20% testing samples utilizing a random state of 42. Standardize the features, and explicitly convert the resulting scaled NumPy arrays back into Pandas DataFrames to retain the feature names.

**Hint:** Execute the standard train-test partition logic. Initialize a standard scaling object, fit it solely on the training predictors, and transform both sets. Reconstruct the output arrays using the `columns` parameter mapped to your original predictor names.

**Expected Output:** The execution of the cell without error, followed by a preview of the scaled training data formatted strictly as a Pandas DataFrame with valid column headers.

In [ ]:
# Write your answer here

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Separate input features and target
X = df.drop(columns=[
    "Machine failure", "TWF", "HDF", "PWF", "OSF", "RNF"
])
y = df["Machine failure"]

# Split into 80% training and 20% testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert NumPy arrays back into Pandas DataFrames
X_train_scaled = pd.DataFrame(
    X_train_scaled, columns=X.columns, index=X_train.index
)
X_test_scaled = pd.DataFrame(
    X_test_scaled, columns=X.columns, index=X_test.index
)

# Display the scaled training data
print(X_train_scaled.head())


      Air temperature [K]  Process temperature [K]  Rotational speed [rpm]  \
9254            -0.854066                -0.609589                0.427634   
1561            -0.904014                -1.080528               -0.834945   
1670            -0.904014                -1.484190               -0.059677   
6087             0.444571                 0.534121                0.333495   
6669             0.694309                 0.332290                0.178441   

      Torque [Nm]  Tool wear [min]    Type_H    Type_L    Type_M  \
9254    -0.892696         1.375035 -0.333796  0.821609 -0.658943   
1561     1.382187         0.457620 -0.333796  0.821609 -0.658943   
1670    -0.892696         1.359218 -0.333796  0.821609 -0.658943   
6087    -0.702288        -1.598655 -0.333796 -1.217123  1.517582   
6669    -0.612094         1.580663 -0.333796  0.821609 -0.658943   

      Temperature_Difference  Power_Estimate  
9254                0.806356       -0.954812  
1561                0.204476

### Question 8: Baseline Logistic Regression Training

Construct a baseline Logistic Regression estimator using the standardized training features. Ensure the algorithm accounts for the target imbalance by applying automated class weighting. Define a maximum iteration limit of 1000 to ensure convergence.

**Hint:** Instantiate the linear classification algorithm with parameters designated for balanced class weighting and extended iterations. Execute the training phase on the processed training subsets.

**Expected Output:** The programmatic instantiation and fitting of a logistic regression object with the specified configuration.

In [ ]:
# Write your answer here
from sklearn.linear_model import LogisticRegression

# Create the Logistic Regression model
model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000
)

# Train the model using training data
model.fit(X_train_scaled, y_train)

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


### Question 9: Cross-Validation and Baseline Evaluation

Validate the stability of your baseline model by executing a 5-fold cross-validation routine on the training set, utilizing ROC-AUC as the primary scoring mechanism. Print the mean cross-validation score. Following this, apply the model to predict outcomes on the test set and print the comprehensive classification report.

**Hint:** Employ a cross-validation function specifying 5 folds and the ROC-AUC metric. Compute the mathematical mean of the resulting array. Extract predictions from the test set and pass them into the standard classification reporting function.

**Expected Output:** A floating-point number representing the mean ROC-AUC across the five training folds, followed by a structured text report detailing precision, recall, and F1-scores for both operating states on the test set.

In [ ]:
# Write your answer here

from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report

# Perform 5-fold cross-validation using ROC-AUC
cv_scores = cross_val_score(
    model,
    X_train_scaled,
    y_train,
    cv=5,
    scoring="roc_auc"
)

# Print the mean cross-validation ROC-AUC score
print("Mean Cross-Validation ROC-AUC:", cv_scores.mean())

# Predict outcomes on the test set
y_pred = model.predict(X_test_scaled)

# Print the classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Mean Cross-Validation ROC-AUC: 0.9239508781101857

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.86      0.92      1939
           1       0.15      0.79      0.26        61

    accuracy                           0.86      2000
   macro avg       0.57      0.83      0.59      2000
weighted avg       0.97      0.86      0.90      2000



### Question 10: Hyperparameter Tuning via GridSearchCV

Optimize the Logistic Regression estimator by conducting an exhaustive grid search. Define a grid to tune the inverse regularization strength parameter ('C') across the values [0.01, 0.1, 1, 10]. Configure the search to use 5-fold cross-validation and ROC-AUC scoring. Output the optimal parameters identified by the search.

**Hint:** Construct a parameter dictionary mapping the exact parameter name to the list of candidate floats. Initialize the grid search object with the baseline model, the grid, the cross-validation strategy, and the scoring metric. Fit the search space using the training data and extract the optimal parameter attribute.

**Expected Output:** A programmatic confirmation of the best estimator configuration, specifically indicating the ideal scalar value for the regularization parameter.

In [ ]:
# Write your answer here

from sklearn.model_selection import GridSearchCV

# Define the parameter grid
param_grid = {
    "C": [0.01, 0.1, 1, 10]
}

# Initialize GridSearchCV using the baseline model
grid_search = GridSearchCV(
    model,
    param_grid,
    cv=5,
    scoring="roc_auc"
)

# Train and evaluate the different parameter values
grid_search.fit(X_train_scaled, y_train)

# Print the best parameter
print("Best Parameters:", grid_search.best_params_)

Best Parameters: {'C': 10}


### Question 11: Multi-Layer Perceptron Initialization

Initialize a Multi-Layer Perceptron (MLP) Classifier to evaluate a non-linear architectural approach. Design the network topography to include two hidden layers consisting of 64 and 32 nodes, respectively. Implement the ReLU activation function, select the Adam optimization solver, restrict training to 500 maximum iterations, and set the random state to 42.

**Hint:** Instantiate the neural network classifier object. Define the layer topography using a tuple format corresponding to the node counts per hidden layer.

**Expected Output:** An MLP classifier variable initialized successfully with the specified architectural and optimization parameters.

In [ ]:
# Write your answer here

from sklearn.neural_network import MLPClassifier

# Initialize the MLP classifier
mlp_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    max_iter=500,
    random_state=42
)

print("MLP Classifier initialized successfully.")

MLP Classifier initialized successfully.


### Question 12: MLP Training and ROC-AUC Evaluation

Fit the Multi-Layer Perceptron architecture to the standardized training data. Extract the predicted probabilities associated with the positive class (failure) for the test set. Calculate and print the resultant Receiver Operating Characteristic Area Under the Curve (ROC-AUC) score.

**Hint:** Execute the training phase. Utilize the probability prediction method and index the correct column to isolate positive class estimates. Compute the ROC-AUC score by evaluating these continuous probabilities against the true binary labels.

**Expected Output:** A single continuous floating-point value reflecting the discriminative capacity (ROC-AUC) of the neural network on the unseen testing data.

In [ ]:
# Write your answer here

from sklearn.metrics import roc_auc_score

# Train the MLP model
mlp_model.fit(X_train_scaled, y_train)

# Predict probabilities for the test data
y_prob = mlp_model.predict_proba(X_test_scaled)[:, 1]

# Calculate and print ROC-AUC score
roc_auc = roc_auc_score(y_test, y_prob)

print("MLP ROC-AUC Score:", roc_auc)


MLP ROC-AUC Score: 0.9711698610911489


### Question 13: Custom Business Metric Implementation

In industrial contexts, error costs are asymmetric. Assume an unexpected equipment failure (False Negative) costs a facility $10,000 in downtime, while an unnecessary maintenance check (False Positive) costs $500. Construct a custom Python function to compute the total financial penalty. Evaluate and print this total cost using the default binary predictions of the trained MLP model.

**Hint:** Formulate a function `calculate_financial_penalty(y_true, y_pred)`. Within the function, utilize an evaluation matrix tool to extract true negatives, false positives, false negatives, and true positives. Structure your logic to return the algebraic sum of the false negative count multiplied by 10000 and the false positive count multiplied by 500.

**Expected Output:** A discrete monetary figure representing the aggregate financial loss incurred by the model's misclassifications on the test dataset.

In [ ]:
# Write your answer here

from sklearn.metrics import confusion_matrix

# Create a function to calculate the total financial penalty
def calculate_financial_penalty(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    total_cost = (fn * 10000) + (fp * 500)

    return total_cost

# Generate default binary predictions from the trained MLP model
y_pred_mlp = mlp_model.predict(X_test_scaled)

# Calculate and print the total financial penalty
total_penalty = calculate_financial_penalty(y_test, y_pred_mlp)

print("Total Financial Penalty: $", total_penalty)


Total Financial Penalty: $ 235000


### Question 14: Decision Threshold Optimization

The default probability threshold for classification is 0.5. Programmatically iterate through candidate thresholds from 0.1 to 0.9 inclusive, in intervals of 0.1. Identify the specific threshold that minimizes the custom financial penalty calculated in the previous question, based on the MLP probabilities. Print the optimal threshold and its corresponding minimum cost.

**Hint:** Establish tracking variables for the minimum cost and the optimal threshold. Construct an iterative loop through the candidate values. During each iteration, apply a logical operator to the continuous probabilities to generate a temporary binary array. Pass this array into your custom cost function, updating the tracking variables whenever a lower penalty is observed.

**Expected Output:** Two distinct numeric outputs: the optimized continuous probability boundary, and the minimized total financial cost achieved by calibrating the decision threshold.

In [ ]:
# Write your answer here

# Initialize the minimum cost and optimal threshold
min_cost = float("inf")
optimal_threshold = 0.1

# Test thresholds from 0.1 to 0.9 in intervals of 0.1
for threshold in [i / 10 for i in range(1, 10)]:

    # Generate binary predictions using the current threshold
    y_pred_threshold = (y_prob >= threshold).astype(int)

    # Calculate the financial penalty
    cost = calculate_financial_penalty(y_test, y_pred_threshold)

    # Update the minimum cost and optimal threshold
    if cost < min_cost:
        min_cost = cost
        optimal_threshold = threshold

# Print the optimal threshold and its minimum financial cost
print("Optimal Threshold:", optimal_threshold)
print("Minimum Financial Penalty: $", min_cost)


Optimal Threshold: 0.1
Minimum Financial Penalty: $ 170500


### Question 15: Permutation Importance & Analytical Reasoning

Determine the driving factors of your MLP model by executing a permutation importance algorithm on the test data. Configure the function to perform 5 repeats utilizing the ROC-AUC scoring metric. Print the top three most influential features based on the mean importance score. Finally, provide a written analysis explaining why permutation importance offers a more rigorous evaluation of feature significance than the basic linear correlation calculated in Question 6.

**Hint:** Utilize standard inspection modules to extract the `importances_mean` array. Map these scores to the corresponding predictor identifiers and sort them. Print your written answer using Python `print()` statements.

**Expected Output:** A ranked output detailing the three most critical features for the neural network, followed by a formal text explanation distinguishing non-linear algorithmic dependency from basic bivariate linear correlation.

In [ ]:
# Write your answer here


from sklearn.inspection import permutation_importance
import pandas as pd

# Calculate permutation importance
perm_importance = permutation_importance(
    mlp_model,
    X_test_scaled,
    y_test,
    n_repeats=5,
    scoring="roc_auc",
    random_state=42
)

# Store feature names and their mean importance scores
feature_importance = pd.DataFrame({
    "Feature": X_test_scaled.columns,
    "Importance": perm_importance.importances_mean
})

# Sort features by importance
feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

# Print the top three most important features
print("Top 3 Most Influential Features:")
print(feature_importance.head(3).to_string(index=False))

# Print the analytical explanation
print("\nAnalytical Explanation:")
print("Permutation importance helps us find which features are most useful for predicting machine failure. It works by randomly shuffling one feature at a time and checking how much the model's ROC-AUC score decreases. A larger decrease means that the feature is more important to the model. Pearson correlation, used in Question 6, measures only the linear relationship between a feature and machine failure. Permutation importance is more useful for our MLP model because it can identify features that help the model make predictions through nonlinear relationships as well.")


Top 3 Most Influential Features:
               Feature  Importance
           Torque [Nm]    0.406784
Temperature_Difference    0.134037
       Tool wear [min]    0.133848

Analytical Explanation:
Permutation importance helps us find which features are most useful for predicting machine failure. It works by randomly shuffling one feature at a time and checking how much the model's ROC-AUC score decreases. A larger decrease means that the feature is more important to the model. Pearson correlation, used in Question 6, measures only the linear relationship between a feature and machine failure. Permutation importance is more useful for our MLP model because it can identify features that help the model make predictions through nonlinear relationships as well.
